In [291]:
import pandas as pd

# NAIVE_BAYES MOVIE REVIEW

In [293]:
from sklearn.naive_bayes import MultinomialNB

In [294]:
df=pd.read_csv(r"C:\Users\vv\Downloads\IMDB Dataset.csv")
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [295]:
# import re

# def clean_review(text):
#     text = str(text).lower()
#     # Corrected: Added missing =, added closing bracket in regex, and closed the quote
#     text = re.sub(r'\[.*?\]', '', text)
#     # Corrected: Fixed extra comma and properly formatted the substitution
#     text = re.sub(r'https?://\S+|www\.\S+', '', text)
#     # Corrected: Properly closed the re.sub call
#     text = re.sub(r'<.*?>+', '', text)
#     text = emoji.replace_emoji(text, replace='')
#     text = re.sub(r'\d+', '', text)
#     text = re.sub(r'[^a-zA-Z\s]', ' ', text)
#     text = re.sub(r'\s+', ' ', text).strip()
#     text = re.sub(f'[{re.escape(string.punctuation)}]', '', text)
#     stop_words = set(stopwords.words('english'))
#     tokens = word_tokenize(text)
#     cleaned_tokens = [w for w in tokens if not w in stop_words]
#     return text

In [296]:
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

def clean(text):
    # 1. Lowercase
    text = text.lower()
    # 2. Remove HTML
    text = re.sub(r'<.*?>', '', text)
    # 3. Remove Smileys and Symbols (keeping only alphanumeric and spaces)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    # 4. Remove numbers
    text = re.sub(r'\d+', '', text)
    
    text = re.sub(r'[?|!|\'|"|#]',r'',text)
    text = re.sub(r'[.|,|)|(|\|/]',r' ',text)
     # 3. Remove URLs
    text = re.sub(r'http\S+', '', text)
    
    # Tokenization and removal of stop words
    tokens = text.split()
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    
    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return " ".join(tokens)

In [297]:
df['review']=df['review'].apply(clean)
df

,review,sentiment
0,one reviewer mentioned watching oz episode you...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically there family little boy jake think t...,negative
4,petter matteis love time money visually stunni...,positive
...,...,...
49995,thought movie right good job wasnt creative or...,positive
49996,bad plot bad dialogue bad acting idiotic direc...,negative
49997,catholic taught parochial elementary school nu...,negative
49998,im going disagree previous comment side maltin...,negative


In [298]:
from nltk.stem import PorterStemmer
def simple_stemmer(text):
    ps = PorterStemmer()
    text= ' '.join([ps.stem(word) for word in text.split()])
    return text

In [299]:
df['review'] = df['review'].apply(simple_stemmer)

In [300]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [301]:
df.columns

Index(['review', 'sentiment'], dtype='object')

In [302]:
from sklearn.preprocessing import LabelEncoder
encoder=LabelEncoder()
df['sentiment']=encoder.fit_transform(df.sentiment)

In [303]:
df

,review,sentiment
0,one review mention watch oz episod youll hook ...,1
1,wonder littl product film techniqu unassum old...,1
2,thought wonder way spend time hot summer weeke...,1
3,basic there famili littl boy jake think there ...,0
4,petter mattei love time money visual stun film...,1
...,...,...
49995,thought movi right good job wasnt creativ orig...,1
49996,bad plot bad dialogu bad act idiot direct anno...,0
49997,cathol taught parochi elementari school nun ta...,0
49998,im go disagre previou comment side maltin one ...,0


In [304]:
x=df.iloc[0:,0]
x

0        one review mention watch oz episod youll hook ...
1        wonder littl product film techniqu unassum old...
2        thought wonder way spend time hot summer weeke...
3        basic there famili littl boy jake think there ...
4        petter mattei love time money visual stun film...
                               ...                        
49995    thought movi right good job wasnt creativ orig...
49996    bad plot bad dialogu bad act idiot direct anno...
49997    cathol taught parochi elementari school nun ta...
49998    im go disagre previou comment side maltin one ...
49999    one expect star trek movi high art fan expect ...
Name: review, Length: 50000, dtype: object

In [305]:
y=df.iloc[0:,1]
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 50000, dtype: int64

In [306]:
from sklearn.model_selection import train_test_split


x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.1, random_state=42)

In [307]:
# from sklearn.feature_extraction.text  import CountVectorizer
# vector=CountVectorizer(stop_words='english', max_features=7000,ngram_range=(1, 2),min_df=5,max_df=0.7)  # Uni-grams and Bi-grams
# vector.fit(x_train)

In [318]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Replace CountVectorizer with this:
vector = TfidfVectorizer(
    stop_words='english',
    max_features=100000, # Keep high to capture nuances
    ngram_range=(1, 3),  # Added trigrams for context
    min_df=5,            # Removed very rare words
    max_df=0.5,          # Removed very common words
    sublinear_tf=True    # Apply logarithmic scaling
)
vector.fit(x_train)

TfidfVectorizer(max_df=0.5, max_features=100000, min_df=5, ngram_range=(1, 3),
                stop_words='english', sublinear_tf=True)

In [319]:
# from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
# from sklearn.naive_bayes import MultinomialNB
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import accuracy_score

In [320]:
# pipeline = Pipeline([
#     ('vect', CountVectorizer(stop_words='english', ngram_range=(1, 2),min_df=5,max_features=50000)),
#     ('tfidf', TfidfTransformer()), # Weighting improves accuracy over raw counts
#     ('clf', MultinomialNB(alpha=0.1))]) # Laplace Smoothing tuned to 0.1 for high precision

In [321]:
# x_train, x_test, y_train, y_test = train_test_split(df['review'], df['sentiment'], test_size=0.1,random_state=10)
# pipeline.fit(x_train, y_train)

In [322]:
# predictions = pipeline.predict(x_test)
# print(f"Accuracy: {accuracy_score(y_test, predictions) * 100:.2f}%")

In [323]:
# # Fit the entire pipeline (transformers + model)
# pipeline.fit(x_train, y_train)

# # Use predict to get outcomes
# predictions = pipeline.predict(x_test)

In [324]:
x_train_transformed = vector.transform(x_train)
x_test_transformed = vector.transform(x_test)

In [325]:
from sklearn.naive_bayes import MultinomialNB
model=MultinomialNB(alpha=0.1)

model.fit(x_train_transformed, y_train)

y_pred=model.predict(x_test_transformed)

y_pred_prob=model.predict_proba(x_test_transformed)
y_pred_prob

array([[0.43804227, 0.56195773],
       [0.09246826, 0.90753174],
       [0.97152301, 0.02847699],
       ...,
       [0.78019843, 0.21980157],
       [0.06587926, 0.93412074],
       [0.79328175, 0.20671825]], shape=(5000, 2))

In [326]:
from sklearn.metrics import confusion_matrix, accuracy_score

results=confusion_matrix(y_test,y_pred)
print('confusion matrix')

print('Accuracy score:', accuracy_score(y_test, y_pred))

confusion matrix
Accuracy score: 0.8828


In [327]:
# t3="one of the other reviewers has mentioned that ...	"
# text1=vector.transform([t3])
# prd=model.predict(text1)
# original=encoder.inverse_transform(prd)
# original[0]